In [ ]:
import zipfile
import shutil
from pathlib import Path
from xml.etree import ElementTree as ET

# Change to project directory
import os
os.chdir('e:\\BCA Final Year Project\\spring-boot-quiz-app-main')

def add_text_to_slide(root, text_to_add, ns):
    """Add text to an existing text box or create new paragraph"""
    text_bodies = root.findall('.//a:txBody', ns)
    if text_bodies:
        last_txbody = text_bodies[-1]
        p_elem = ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/main}p')
        r_elem = ET.SubElement(p_elem, '{http://schemas.openxmlformats.org/drawingml/2006/main}r')
        rPr = ET.SubElement(r_elem, '{http://schemas.openxmlformats.org/drawingml/2006/main}rPr')
        rPr.set('lang', 'en-US')
        rPr.set('dirty', '0')
        t_elem = ET.SubElement(r_elem, '{http://schemas.openxmlformats.org/drawingml/2006/main}t')
        t_elem.text = text_to_add
        last_txbody.append(p_elem)
        return True
    return False

pptx_path = Path('QUIZ_PLATFORM.pptx')
backup_path = Path('QUIZ_PLATFORM_backup.pptx')
shutil.copy(pptx_path, backup_path)
print(f'✓ Backed up to {backup_path}')

extract_dir = Path('pptx_temp')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
with zipfile.ZipFile(pptx_path, 'r') as z:
    z.extractall(extract_dir)
print(f'✓ Extracted PPTX to {extract_dir}')

NS = {
    'a': 'http://schemas.openxmlformats.org/drawingml/2006/main',
    'p': 'http://schemas.openxmlformats.org/presentationml/2006/main',
    'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'
}

# Update Slide 5 (Process Logic)
slide5_path = extract_dir / 'ppt/slides/slide5.xml'
tree5 = ET.parse(slide5_path)
root5 = tree5.getroot()
add_text_to_slide(root5, 'Admin Panel Flow:', NS)
add_text_to_slide(root5, 'Admin Login → Dashboard → Manage Questions → View Results', NS)
add_text_to_slide(root5, 'Add/Edit/Delete Questions, Monitor Performance', NS)
tree5.write(slide5_path, encoding='utf-8', xml_declaration=True)
print('✓ Updated Slide 5: Process Logic with Admin Workflow')

# Update Slide 10 (ER Diagram)
slide10_path = extract_dir / 'ppt/slides/slide10.xml'
tree10 = ET.parse(slide10_path)
root10 = tree10.getroot()
add_text_to_slide(root10, 'Admin Role: Manages Question & Result Entities', NS)
add_text_to_slide(root10, 'Authentication protects admin operations', NS)
tree10.write(slide10_path, encoding='utf-8', xml_declaration=True)
print('✓ Updated Slide 10: ER Diagram with Admin Role')

# Update Slide 11 (Interface)
slide11_path = extract_dir / 'ppt/slides/slide11.xml'
tree11 = ET.parse(slide11_path)
root11 = tree11.getroot()
add_text_to_slide(root11, 'Admin Interface Screens:', NS)
add_text_to_slide(root11, 'Admin Login • Dashboard • Question Manager • Results Viewer', NS)
tree11.write(slide11_path, encoding='utf-8', xml_declaration=True)
print('✓ Updated Slide 11: Interface with Admin Screens')

# Repackage PPTX
output_pptx = Path('QUIZ_PLATFORM_UPDATED.pptx')
if output_pptx.exists():
    output_pptx.unlink()

def zip_directory(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in folder_path.walk():
            for file in files:
                file_path = root / file
                arcname = file_path.relative_to(folder_path)
                zipf.write(file_path, arcname)

zip_directory(extract_dir, output_pptx)
print(f'✓ Created updated PPTX: {output_pptx}')

# Cleanup
shutil.rmtree(extract_dir)
print('\n✅ PPT Update Complete!')
print(f'📁 Updated file: {output_pptx.absolute()}')


In [ ]:
import zipfile
import shutil
import os
from pathlib import Path

os.chdir(r'e:\BCA Final Year Project\spring-boot-quiz-app-main')

# Paths
pptx_path = 'QUIZ_PLATFORM.pptx'
output_path = 'QUIZ_PLATFORM_UPDATED.pptx'
backup_path = 'QUIZ_PLATFORM_BACKUP.pptx'
temp_extract = 'pptx_temp_extract'

# Create backup
if not os.path.exists(backup_path):
    shutil.copy(pptx_path, backup_path)
    print(f'✓ Backup created: {backup_path}')

# Extract
if os.path.exists(temp_extract):
    shutil.rmtree(temp_extract)
os.makedirs(temp_extract)

with zipfile.ZipFile(pptx_path, 'r') as z:
    z.extractall(temp_extract)

print('✓ Extracted PPTX')

# Content to append
updates = {
    'ppt/slides/slide5.xml': '\nAdmin Workflow: Login → Dashboard → Question Mgmt (Add/Edit/Delete) → DB Update → Results Review',
    'ppt/slides/slide10.xml': '\nAdmin Management: Question & Result entities with direct CRUD admin operations',
    'ppt/slides/slide11.xml': '\nAdmin Screens: Login | Dashboard | Question Manager | Results Viewer'
}

# Update slides
for slide, text_to_add in updates.items():
    slide_path = os.path.join(temp_extract, slide)
    if os.path.exists(slide_path):
        with open(slide_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Find last </a:txBody> and insert new paragraph
        last_txbody = content.rfind('</a:txBody>')
        if last_txbody > 0:
            new_para = f'<a:p><a:r><a:rPr lang="en-US" sz="1800" dirty="0"/><a:t>{text_to_add}</a:t></a:r></a:p>'
            content = content[:last_txbody] + new_para + content[last_txbody:]
            
            with open(slide_path, 'w', encoding='utf-8') as f:
                f.write(content)
            
            print(f'✓ Updated: {slide}')
    else:
        print(f'⚠ Not found: {slide}')

# Repackage
if os.path.exists(output_path):
    os.remove(output_path)

with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(temp_extract):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, temp_extract)
            z.write(file_path, arcname)

print(f'✓ Created: {output_path}')

# Cleanup
shutil.rmtree(temp_extract)
print('✓ Complete!')


# Admin Panel Documentation

This notebook summarizes the missing admin panel details for the Quiz Platform project and provides a focused set of content to append without altering the existing presentation design or content.

## 1. Admin Panel Overview
The admin panel is the secure backend interface used by administrators to manage quiz content, review results, and maintain the platform. It supports centralized question management, result monitoring, and system configuration, making it a critical part of the project architecture.

## 2. Admin User and Role Management
The platform includes a dedicated admin user with a secure login. Administrators have elevated permissions to add and remove quiz questions, view candidate results, and manage quiz data. In this project, the admin role is responsible for:

- Logging in with admin credentials.
- Managing questions stored in the MySQL database.
- Viewing student results and interaction metrics.

## 3. Quiz and Question Management from Admin Interface
Administrators can add new quiz questions, delete existing questions, and update question content. The admin interface supports:

- Creating new questions with options and the correct answer.
- Deleting questions when they are no longer needed.
- Maintaining question categories to organize quizzes.

The admin panel writes quiz data directly to the database, ensuring the quiz content is kept up-to-date and can be used immediately by the quiz application.

## 4. Analytics and Reports for Admin
The admin panel includes reporting functionality that allows administrators to:

- View the latest quiz results.
- Monitor student participation and performance.
- See the total number of questions and attempted quizzes.

The dashboard summarizes high-level metrics, including the number of students attempted and total results captured in the database.

## 5. Security and Access Control for Admin Panel
The admin panel is protected by authentication to ensure only authorized users can access it. Security measures include:

- Requiring admin login credentials to access admin routes.
- Invalidating sessions on logout.
- Preventing cached pages with no-cache headers.

This protects sensitive operations like question modification and result access.

## 6. Updated Process Logic
The admin panel adds a second workflow to the existing quiz system. In addition to student quiz execution, the process now includes:

- Admin login and session validation.
- Access to admin dashboards for question and result management.
- CRUD operations for quiz questions.
- Real-time updates of the quiz database so new questions are immediately available to students.

This means the process logic flow covers both student quiz participation and administrative question management, ensuring both paths are clearly represented.

## 7. Updated ER Diagram
The ER diagram remains centered on the `Question` and `Result` entities, but now also reflects the admin management role:

- `Question` stores `title`, `optionA`, `optionB`, `optionC`, `ans`, and `category`.
- `Result` stores `username`, `score`, `category`, and quiz metadata.
- The admin interface manages `Question` records and reviews `Result` records.

The admin panel acts as a management layer over these entities, with `Question` and `Result` connected through the quiz workflow and administrative dashboard.
